In [1]:
import re
import os

In [2]:
print(os.getcwd())

d:\MSc Artificial Intelligence\COMP5012


In [3]:
file_path = "data/Modules (data for JAN assessment).txt"

In [4]:
def normalise_module_code(code: str) -> str:
    code = code.strip().upper()
    code = code.replace("M0D", "MOD")

    import re
    match = re.match(r"MOD(\d+)$", code)
    if match:
        return f"MOD{match.group(1).zfill(3)}"

    digits = re.findall(r"\d+", code)
    if digits:
        return f"MOD{digits[0].zfill(3)}"

    return code


def parse_module_line(line: str) -> dict:
    parts = line.strip().split("|")

    module_code = normalise_module_code(parts[0])
    staff_name = parts[1].strip()
    num_labs = int(parts[2].strip())

    conflicts = [
        normalise_module_code(c)
        for c in parts[3].split(",")
        if c.strip()
    ]

    return {
        "module_id": module_code,
        "staff": staff_name,
        "num_labs": num_labs,
        "conflicts": conflicts
    }


def load_modules(file_path):
    modules = []

    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                modules.append(parse_module_line(line))

    return modules

In [5]:
modules = load_modules(file_path)

print(f"Loaded {len(modules)} modules")
modules[:3]

Loaded 17 modules


[{'module_id': 'MOD001',
  'staff': 'Zacharias Karstensen',
  'num_labs': 2,
  'conflicts': ['MOD002',
   'MOD003',
   'MOD004',
   'MOD005',
   'MOD006',
   'MOD007',
   'MOD008',
   'MOD009',
   'MOD010',
   'MOD013']},
 {'module_id': 'MOD002',
  'staff': 'Dominykas Cleary',
  'num_labs': 2,
  'conflicts': ['MOD001',
   'MOD003',
   'MOD004',
   'MOD005',
   'MOD006',
   'MOD007',
   'MOD008',
   'MOD009',
   'MOD010',
   'MOD013']},
 {'module_id': 'MOD003',
  'staff': 'Zacharias Karstensen',
  'num_labs': 2,
  'conflicts': ['MOD001',
   'MOD002',
   'MOD004',
   'MOD005',
   'MOD006',
   'MOD007',
   'MOD008',
   'MOD009',
   'MOD010',
   'MOD011',
   'MOD012',
   'MOD013']}]

In [6]:
def build_events(modules):
    """
    Convert module data into a flat list of teaching events.
    
    Each module creates:
    - 1 lecture event
    - num_labs lab events
    """
    events = []

    for module in modules:
        module_id = module["module_id"]
        staff = module["staff"]

        # Add lecture event
        events.append({
            "event_id": f"{module_id}_LEC",
            "module_id": module_id,
            "staff": staff,
            "event_type": "lecture"
        })

        # Add lab events
        for lab_num in range(1, module["num_labs"] + 1):
            events.append({
                "event_id": f"{module_id}_LAB{lab_num}",
                "module_id": module_id,
                "staff": staff,
                "event_type": "lab"
            })

    return events

In [7]:
events = build_events(modules)

print(f"Total number of events: {len(events)}")
events[:10]

Total number of events: 48


[{'event_id': 'MOD001_LEC',
  'module_id': 'MOD001',
  'staff': 'Zacharias Karstensen',
  'event_type': 'lecture'},
 {'event_id': 'MOD001_LAB1',
  'module_id': 'MOD001',
  'staff': 'Zacharias Karstensen',
  'event_type': 'lab'},
 {'event_id': 'MOD001_LAB2',
  'module_id': 'MOD001',
  'staff': 'Zacharias Karstensen',
  'event_type': 'lab'},
 {'event_id': 'MOD002_LEC',
  'module_id': 'MOD002',
  'staff': 'Dominykas Cleary',
  'event_type': 'lecture'},
 {'event_id': 'MOD002_LAB1',
  'module_id': 'MOD002',
  'staff': 'Dominykas Cleary',
  'event_type': 'lab'},
 {'event_id': 'MOD002_LAB2',
  'module_id': 'MOD002',
  'staff': 'Dominykas Cleary',
  'event_type': 'lab'},
 {'event_id': 'MOD003_LEC',
  'module_id': 'MOD003',
  'staff': 'Zacharias Karstensen',
  'event_type': 'lecture'},
 {'event_id': 'MOD003_LAB1',
  'module_id': 'MOD003',
  'staff': 'Zacharias Karstensen',
  'event_type': 'lab'},
 {'event_id': 'MOD003_LAB2',
  'module_id': 'MOD003',
  'staff': 'Zacharias Karstensen',
  'event_t

In [8]:
for event in events:
    print(event)

{'event_id': 'MOD001_LEC', 'module_id': 'MOD001', 'staff': 'Zacharias Karstensen', 'event_type': 'lecture'}
{'event_id': 'MOD001_LAB1', 'module_id': 'MOD001', 'staff': 'Zacharias Karstensen', 'event_type': 'lab'}
{'event_id': 'MOD001_LAB2', 'module_id': 'MOD001', 'staff': 'Zacharias Karstensen', 'event_type': 'lab'}
{'event_id': 'MOD002_LEC', 'module_id': 'MOD002', 'staff': 'Dominykas Cleary', 'event_type': 'lecture'}
{'event_id': 'MOD002_LAB1', 'module_id': 'MOD002', 'staff': 'Dominykas Cleary', 'event_type': 'lab'}
{'event_id': 'MOD002_LAB2', 'module_id': 'MOD002', 'staff': 'Dominykas Cleary', 'event_type': 'lab'}
{'event_id': 'MOD003_LEC', 'module_id': 'MOD003', 'staff': 'Zacharias Karstensen', 'event_type': 'lecture'}
{'event_id': 'MOD003_LAB1', 'module_id': 'MOD003', 'staff': 'Zacharias Karstensen', 'event_type': 'lab'}
{'event_id': 'MOD003_LAB2', 'module_id': 'MOD003', 'staff': 'Zacharias Karstensen', 'event_type': 'lab'}
{'event_id': 'MOD004_LEC', 'module_id': 'MOD004', 'staff':

In [ ]:
num_lectures = sum(1 for event in events if event["event_type"] == "lecture")
num_labs = sum(1 for event in events if event["event_type"] == "lab")

print("Number of lectures:", num_lectures)
print("Number of labs:", num_labs)
print("Total events:", len(events))

Number of lectures: 17
Number of labs: 31
Total events: 48


In [10]:
DAYS_PER_WEEK = 5
SLOTS_PER_DAY = 4
TOTAL_SLOTS = DAYS_PER_WEEK * SLOTS_PER_DAY

DAY_NAMES = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]
TIME_LABELS = ["09:00-11:00", "11:00-13:00", "14:00-16:00", "16:00-18:00"]

print("Total timetable slots:", TOTAL_SLOTS)

Total timetable slots: 20


In [11]:
def slot_to_day(slot):
    """
    Convert a slot number into a day index.
    Example:
    0-3   -> Monday
    4-7   -> Tuesday
    8-11  -> Wednesday
    12-15 -> Thursday
    16-19 -> Friday
    """
    return slot // SLOTS_PER_DAY


def slot_to_time_index(slot):
    """
    Convert a slot number into a time index within the day.
    """
    return slot % SLOTS_PER_DAY


def slot_to_label(slot):
    """
    Convert a slot number into a readable label like:
    'Monday 09:00-11:00'
    """
    day_name = DAY_NAMES[slot_to_day(slot)]
    time_label = TIME_LABELS[slot_to_time_index(slot)]
    return f"{day_name} {time_label}"

In [12]:
for slot in range(TOTAL_SLOTS):
    print(slot, "->", slot_to_label(slot))

0 -> Monday 09:00-11:00
1 -> Monday 11:00-13:00
2 -> Monday 14:00-16:00
3 -> Monday 16:00-18:00
4 -> Tuesday 09:00-11:00
5 -> Tuesday 11:00-13:00
6 -> Tuesday 14:00-16:00
7 -> Tuesday 16:00-18:00
8 -> Wednesday 09:00-11:00
9 -> Wednesday 11:00-13:00
10 -> Wednesday 14:00-16:00
11 -> Wednesday 16:00-18:00
12 -> Thursday 09:00-11:00
13 -> Thursday 11:00-13:00
14 -> Thursday 14:00-16:00
15 -> Thursday 16:00-18:00
16 -> Friday 09:00-11:00
17 -> Friday 11:00-13:00
18 -> Friday 14:00-16:00
19 -> Friday 16:00-18:00


In [13]:
import random

In [14]:
def create_random_chromosome(events, total_slots=TOTAL_SLOTS):
    """
    Create one random timetable chromosome.
    Each gene is a slot number assigned to one event.
    """
    chromosome = [random.randint(0, total_slots - 1) for _ in events]
    return chromosome

In [15]:
chromosome = create_random_chromosome(events)

print("Chromosome length:", len(chromosome))
print("First 10 genes:", chromosome[:10])

Chromosome length: 48
First 10 genes: [6, 1, 19, 17, 5, 10, 13, 11, 10, 5]


In [16]:
def decode_chromosome(chromosome, events):
    """
    Convert a chromosome into a readable list of event assignments.
    """
    decoded = []

    for event, slot in zip(events, chromosome):
        decoded.append({
            "event_id": event["event_id"],
            "module_id": event["module_id"],
            "staff": event["staff"],
            "event_type": event["event_type"],
            "slot": slot,
            "slot_label": slot_to_label(slot)
        })

    return decoded

In [17]:
decoded = decode_chromosome(chromosome, events)

for row in decoded[:10]:
    print(row)

{'event_id': 'MOD001_LEC', 'module_id': 'MOD001', 'staff': 'Zacharias Karstensen', 'event_type': 'lecture', 'slot': 6, 'slot_label': 'Tuesday 14:00-16:00'}
{'event_id': 'MOD001_LAB1', 'module_id': 'MOD001', 'staff': 'Zacharias Karstensen', 'event_type': 'lab', 'slot': 1, 'slot_label': 'Monday 11:00-13:00'}
{'event_id': 'MOD001_LAB2', 'module_id': 'MOD001', 'staff': 'Zacharias Karstensen', 'event_type': 'lab', 'slot': 19, 'slot_label': 'Friday 16:00-18:00'}
{'event_id': 'MOD002_LEC', 'module_id': 'MOD002', 'staff': 'Dominykas Cleary', 'event_type': 'lecture', 'slot': 17, 'slot_label': 'Friday 11:00-13:00'}
{'event_id': 'MOD002_LAB1', 'module_id': 'MOD002', 'staff': 'Dominykas Cleary', 'event_type': 'lab', 'slot': 5, 'slot_label': 'Tuesday 11:00-13:00'}
{'event_id': 'MOD002_LAB2', 'module_id': 'MOD002', 'staff': 'Dominykas Cleary', 'event_type': 'lab', 'slot': 10, 'slot_label': 'Wednesday 14:00-16:00'}
{'event_id': 'MOD003_LEC', 'module_id': 'MOD003', 'staff': 'Zacharias Karstensen', 'ev

In [18]:
import pandas as pd

decoded_df = pd.DataFrame(decoded)
decoded_df.head(10)

,event_id,module_id,staff,event_type,slot,slot_label
0,MOD001_LEC,MOD001,Zacharias Karstensen,lecture,6,Tuesday 14:00-16:00
1,MOD001_LAB1,MOD001,Zacharias Karstensen,lab,1,Monday 11:00-13:00
2,MOD001_LAB2,MOD001,Zacharias Karstensen,lab,19,Friday 16:00-18:00
3,MOD002_LEC,MOD002,Dominykas Cleary,lecture,17,Friday 11:00-13:00
4,MOD002_LAB1,MOD002,Dominykas Cleary,lab,5,Tuesday 11:00-13:00
5,MOD002_LAB2,MOD002,Dominykas Cleary,lab,10,Wednesday 14:00-16:00
6,MOD003_LEC,MOD003,Zacharias Karstensen,lecture,13,Thursday 11:00-13:00
7,MOD003_LAB1,MOD003,Zacharias Karstensen,lab,11,Wednesday 16:00-18:00
8,MOD003_LAB2,MOD003,Zacharias Karstensen,lab,10,Wednesday 14:00-16:00
9,MOD004_LEC,MOD004,Laila Deniau,lecture,5,Tuesday 11:00-13:00


In [19]:
module_conflicts = {
    module["module_id"]: set(module["conflicts"])
    for module in modules
}

module_conflicts

{'MOD001': {'MOD002',
  'MOD003',
  'MOD004',
  'MOD005',
  'MOD006',
  'MOD007',
  'MOD008',
  'MOD009',
  'MOD010',
  'MOD013'},
 'MOD002': {'MOD001',
  'MOD003',
  'MOD004',
  'MOD005',
  'MOD006',
  'MOD007',
  'MOD008',
  'MOD009',
  'MOD010',
  'MOD013'},
 'MOD003': {'MOD001',
  'MOD002',
  'MOD004',
  'MOD005',
  'MOD006',
  'MOD007',
  'MOD008',
  'MOD009',
  'MOD010',
  'MOD011',
  'MOD012',
  'MOD013'},
 'MOD004': {'MOD001',
  'MOD002',
  'MOD003',
  'MOD005',
  'MOD006',
  'MOD007',
  'MOD008',
  'MOD009',
  'MOD010',
  'MOD011',
  'MOD012',
  'MOD013'},
 'MOD005': {'MOD001',
  'MOD002',
  'MOD003',
  'MOD004',
  'MOD006',
  'MOD007',
  'MOD008',
  'MOD009',
  'MOD010',
  'MOD011',
  'MOD012'},
 'MOD006': {'MOD001',
  'MOD002',
  'MOD003',
  'MOD004',
  'MOD005',
  'MOD007',
  'MOD008',
  'MOD009',
  'MOD010',
  'MOD011',
  'MOD012'},
 'MOD007': {'MOD001',
  'MOD002',
  'MOD003',
  'MOD004',
  'MOD005',
  'MOD006',
  'MOD008',
  'MOD009',
  'MOD010',
  'MOD011',
  'MOD014',


In [20]:
def count_clashes(chromosome, events, module_conflicts):
    """
    Count clashes where two events from conflicting modules
    are scheduled in the same slot.
    
    A clash occurs if:
    - the two events belong to different modules
    - the modules are in each other's conflict list
    - both events are assigned the same slot
    """
    clashes = 0
    counted_pairs = set()

    for i in range(len(events)):
        module_i = events[i]["module_id"]
        slot_i = chromosome[i]

        for j in range(i + 1, len(events)):
            module_j = events[j]["module_id"]
            slot_j = chromosome[j]

            if module_i == module_j:
                continue

            # Check if these modules conflict
            if module_j in module_conflicts.get(module_i, set()):
                if slot_i == slot_j:
                    pair = tuple(sorted((module_i, module_j)) + [slot_i])

                    if pair not in counted_pairs:
                        clashes += 1
                        counted_pairs.add(pair)

    return clashes

In [21]:
def count_staff_teaching_days(chromosome, events):
    """
    Count the total number of unique teaching days across all staff.
    Lower is better because staff teaching should be compressed
    into fewer days.
    """
    staff_days = {}

    for event, slot in zip(events, chromosome):
        staff = event["staff"]
        day = slot_to_day(slot)

        if staff not in staff_days:
            staff_days[staff] = set()

        staff_days[staff].add(day)

    total_days = sum(len(days) for days in staff_days.values())
    return total_days

In [22]:
def count_penalties(chromosome, events):
    """
    Count timetable feasibility violations.

    Penalty types:
    1. Same staff assigned to multiple events in the same slot
    2. Same module assigned to multiple events in the same slot
    """
    penalties = 0

    for i in range(len(events)):
        for j in range(i + 1, len(events)):
            same_slot = chromosome[i] == chromosome[j]

            if not same_slot:
                continue

            # Same staff overlap
            if events[i]["staff"] == events[j]["staff"]:
                penalties += 1

            # Same module overlap
            if events[i]["module_id"] == events[j]["module_id"]:
                penalties += 1

    return penalties

In [23]:
def evaluate_chromosome(chromosome, events, module_conflicts, penalty_weight=100):
    """
    Evaluate one timetable chromosome.

    Returns:
    - objective_1: clashes + penalty
    - objective_2: staff teaching days + penalty
    - raw details for analysis
    """
    clashes = count_clashes(chromosome, events, module_conflicts)
    staff_days = count_staff_teaching_days(chromosome, events)
    penalties = count_penalties(chromosome, events)

    objective_1 = clashes + penalty_weight * penalties
    objective_2 = staff_days + penalty_weight * penalties

    return {
        "objective_1": objective_1,
        "objective_2": objective_2,
        "clashes": clashes,
        "staff_days": staff_days,
        "penalties": penalties
    }

In [24]:
fitness = evaluate_chromosome(chromosome, events, module_conflicts)

fitness

{'objective_1': 429,
 'objective_2': 426,
 'clashes': 29,
 'staff_days': 26,
 'penalties': 4}

In [25]:
print("Objective 1 (clashes + penalties):", fitness["objective_1"])
print("Objective 2 (staff days + penalties):", fitness["objective_2"])
print("Raw clashes:", fitness["clashes"])
print("Raw staff teaching days:", fitness["staff_days"])
print("Penalties:", fitness["penalties"])

Objective 1 (clashes + penalties): 429
Objective 2 (staff days + penalties): 426
Raw clashes: 29
Raw staff teaching days: 26
Penalties: 4


In [26]:
for i in range(5):
    test_chromosome = create_random_chromosome(events)
    test_fitness = evaluate_chromosome(test_chromosome, events, module_conflicts)
    print(f"Chromosome {i+1}: {test_fitness}")

Chromosome 1: {'objective_1': 1031, 'objective_2': 1028, 'clashes': 31, 'staff_days': 28, 'penalties': 10}
Chromosome 2: {'objective_1': 1132, 'objective_2': 1127, 'clashes': 32, 'staff_days': 27, 'penalties': 11}
Chromosome 3: {'objective_1': 934, 'objective_2': 927, 'clashes': 34, 'staff_days': 27, 'penalties': 9}
Chromosome 4: {'objective_1': 934, 'objective_2': 924, 'clashes': 34, 'staff_days': 24, 'penalties': 9}
Chromosome 5: {'objective_1': 1526, 'objective_2': 1523, 'clashes': 26, 'staff_days': 23, 'penalties': 15}
